# Create Card Strength Metric

**Workflow:**
- Load and filter raw Scryfall data for Commander-legal cards.
- Fetch and join EDHREC inclusion data (play rate) to serve as training labels.
- Export the final dataset for model training.

In [ ]:
import os
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

from common.dataset_filter import filter_dataset
from common.text_normalizing import sluggify_card_name

In [ ]:
INPUT_DATA_PATH = "../data/scryfall/oracle-cards-20251022090300.json"
OUTPUT_DATA_PATH = "../data/training/card_strength_metric.xlsx"
EDHREC_CARD_DIRECTORY = "../data/edhrec/cards"

In [ ]:
df = pd.read_json(INPUT_DATA_PATH)
df = filter_dataset(df)

print(f"shape: {df.shape}")
df.head()

## Join EDHREC Inclusion Data

- Iterate through cards and match them with scraped EDHREC json files.
- Calculate `inclusion_rate` and `log_inclusion_rate` (Popularity / Potential Decks).
- Drop basic lands and cards without available popularity data.

**TODO (for later):**
- For commanders: use higher play rate out of commander play rate or play rate in 99
- Weight the play rate per bracket (bracket 1 contributes 1, bracket 5 contributes 5)
- Track play rate (or frequency) per commander to determine if a card is niche or broad (boost good niche cards) 

In [ ]:
play_rate_data = []
indices_to_drop = []

for idx, row in df.iterrows():
    name = row["name"]
    print(f"({idx+1}/{len(df)}) Processing {name}...")

    if name in ["Forest", "Mountain", "Swamp", "Island", "Plains"]:
        print(f"  -> Skip basic land, drop row\n")
        indices_to_drop.append(idx)
        continue

    slugs = sluggify_card_name(name)
    if isinstance(slugs, str):
        slugs = [slugs]

    inclusion = potential_decks = inclusion_rate = log_inclusion_rate = 0
    for slug in slugs:
        filename = f"{slug}.json"
        path = os.path.join(EDHREC_CARD_DIRECTORY, filename)
        if not os.path.exists(path):
            continue
        try:
            with open(path, 'r', encoding='utf-8') as f:
                json_dict = json.load(f)
            inclusion = json_dict["container"]["json_dict"]["card"]["inclusion"]
            potential_decks = json_dict["container"]["json_dict"]["card"]["potential_decks"]
            inclusion_rate = np.float64(inclusion) / np.float64(potential_decks)
            log_inclusion_rate = np.log10(inclusion_rate)
            break
        except (KeyError, json.JSONDecodeError):
            continue

    if inclusion == 0:
        print(f"  -> No data found, drop row\n")
        indices_to_drop.append(idx)
        continue

    play_rate_data.append({
        'index': idx,
        'inclusion': inclusion,
        'potential_decks': potential_decks,
        'inclusion_rate': inclusion_rate,
        'log_inclusion_rate': log_inclusion_rate
    })

df = df.drop(index=indices_to_drop)
results_df = pd.DataFrame(play_rate_data).set_index('index')
df = df.join(results_df)

print(f"\nProcessing complete.")
print(f"{len(indices_to_drop)} rows dropped")
print(f"DataFrame shape: {df.shape}")

## Create Power Scores

- Scale log-inclusion rates to a standardized 25-99 range to create `log10_power_score`.
- Visualize the score distribution to ensure data quality.

In [ ]:
# TODO: add card salt score into calculation
MIN_SCORE = 25
MAX_SCORE = 99
log_min = df['log_inclusion_rate'].min()
log_max = df['log_inclusion_rate'].max()
df['log_power_score'] = MIN_SCORE + ((df['log_inclusion_rate'] - log_min) * (MAX_SCORE - MIN_SCORE)) / (log_max - log_min)

print(f"Scores calculated! Range: {df['log_power_score'].min():.4f} to {df['log_power_score'].max():.4f}")
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.histplot(df['log_power_score'], bins=70, kde=True, color='indigo')
plt.title('Distribution of MTG Card Power Scores (25-99)', fontsize=16, fontweight='bold')
plt.show()

## Dataset Export

In [ ]:
print(f"Exporting to {OUTPUT_DATA_PATH}")
df.to_excel(OUTPUT_DATA_PATH, index=False)
print("Done")